# Stage 3 prerequisite — regenerate ASpanFormer sidecars for the reviewed pairs

Rebuilds real ASpanFormer sidecars (`raw_mkpts`/`filtered_mkpts`/etc.) for exactly the
(source, target) pairs listed in `match_manifest_shard1.csv` and `match_manifest_shard2.csv`
— the human-reviewed ground truth. The prior run that produced those CSVs used an older
format and never saved sidecars, so they need regenerating before the MASt3R (B10)
ablation can run: MASt3R must see the same homography-aligned pair VGGT sees, which
requires a real sidecar (see `ablation/MAST3R_ABLATION_METHODOLOGY.md`).

**Deliberately narrow-purpose.** This notebook does not modify `geometry_filter.py`, does
not run fresh retrieval, and writes only to its own dedicated local directory
(`/content/mast3r_ablation_work`) — distinct from both `main.ipynb`'s
`/content/image_similarity_dev` and `stage2_ablation_colab.ipynb`'s `/content/work` — so
it can never disturb either notebook's real outputs.

**Before running**, have these already on Drive at `[DRIVE_ROOT]`:
- `geometry_filter.py`, `ml-aspanformer/` (weights + config, same layout `main.ipynb` uses)
- `pipeline_output/retrieval_manifest.jsonl` (full-corpus retrieval manifest)
- `pipeline_output/match_manifest_shard1.csv`, `pipeline_output/match_manifest_shard2.csv`
- `workingsourcecrops.zip`, `working_targetcrops.zip`

**Output:** `vggt_candidates_manifest.jsonl` + `aspan_sidecars/*.npz` for the reviewed
pairs, plus `candidate_classification.json` (candidate_id → Positive/Negative/Unsure/
Unknown side lookup — `geometry_filter.py` doesn't carry that field through on its own).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install ASpanFormer's own requirements (mirrors main.ipynb Section A)
import subprocess, sys
from pathlib import Path

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

ASPAN_REQS = Path('/content/drive/MyDrive/ImageSimilarity/ml-aspanformer/requirements.txt')
if ASPAN_REQS.exists():
    _pip('-r', str(ASPAN_REQS))
else:
    print('WARNING: ASpanFormer requirements.txt not found at', ASPAN_REQS)
print('Install done.')

In [ ]:
from pathlib import Path

# -- Drive paths (mirrors main.ipynb's ASpanFormer config exactly) -------------
DRIVE_ROOT = Path('/content/drive/MyDrive/ImageSimilarity')

GEOFILTER_SCRIPT_DRIVE = DRIVE_ROOT / 'geometry_filter.py'
ASPAN_ROOT             = DRIVE_ROOT / 'ml-aspanformer'
ASPAN_CONFIG           = ASPAN_ROOT / 'configs/aspan/outdoor/aspan_test.py'
ASPAN_WEIGHTS_DRIVE    = DRIVE_ROOT / 'aspanweights/outdoor.ckpt'

# Pre-staged MASt3R checkpoint (.pth, downloaded by hand from MASt3R's GitHub README
# and uploaded here) -- avoids the flaky live Hugging Face Hub download inside
# AsymmetricMASt3R.from_pretrained(). Optional: if not present, cell-run-mast3r falls
# back to the live HF download, loudly, rather than silently behaving differently.
MAST3R_CKPT_DRIVE = DRIVE_ROOT / 'weights/mast3r_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'

SOURCE_ZIP = DRIVE_ROOT / 'workingsourcecrops.zip'
TARGET_ZIP = DRIVE_ROOT / 'working_targetcrops.zip'

RETRIEVAL_MANIFEST = DRIVE_ROOT / 'pipeline_output/retrieval_manifest.jsonl'
SHARD1_CSV = DRIVE_ROOT / 'pipeline_output/match_manifest_shard1.csv'
SHARD2_CSV = DRIVE_ROOT / 'pipeline_output/match_manifest_shard2.csv'

# -- Dedicated local working dir -- distinct from main.ipynb's
#    /content/image_similarity_dev and stage2_ablation_colab.ipynb's /content/work,
#    so this notebook can never collide with either's real outputs.
LOCAL_ROOT = Path('/content/mast3r_ablation_work')
ASPAN_WEIGHTS_LOCAL = LOCAL_ROOT / 'weights/outdoor.ckpt'
MAST3R_CKPT_LOCAL = LOCAL_ROOT / 'weights/mast3r_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
LOCAL_SOURCE_DIR = LOCAL_ROOT / 'source'
LOCAL_TARGET_DIR = LOCAL_ROOT / 'target'
ASPAN_OUTPUT_DIR = LOCAL_ROOT / 'aspan_output'

MAX_PAIRS = None  # set e.g. 10 for a smoke test before the full ~1,282-pair run

for d in [LOCAL_ROOT, LOCAL_SOURCE_DIR, LOCAL_TARGET_DIR, ASPAN_OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Sanity checks
for label, p in [
    ('geometry_filter.py',    GEOFILTER_SCRIPT_DRIVE),
    ('ml-aspanformer',        ASPAN_ROOT),
    ('ASpan config',          ASPAN_CONFIG),
    ('ASpan weights (Drive)', ASPAN_WEIGHTS_DRIVE),
    ('Retrieval manifest',    RETRIEVAL_MANIFEST),
    ('Shard 1 CSV',           SHARD1_CSV),
    ('Shard 2 CSV',           SHARD2_CSV),
    ('Source zip',            SOURCE_ZIP),
    ('Target zip',            TARGET_ZIP),
]:
    print(f'{label}: {"OK" if p.exists() else "NOT FOUND -- update DRIVE_ROOT paths"}')

print(f'MASt3R checkpoint (Drive, optional pre-stage): '
      f'{"OK" if MAST3R_CKPT_DRIVE.exists() else "NOT FOUND -- cell-run-mast3r will fall back to a live HF Hub download"}')

In [ ]:
import shutil, sys

dst = LOCAL_ROOT / GEOFILTER_SCRIPT_DRIVE.name
shutil.copy(GEOFILTER_SCRIPT_DRIVE, dst)
print(f'Copied {GEOFILTER_SCRIPT_DRIVE.name} -> {dst}')

ASPAN_WEIGHTS_LOCAL.parent.mkdir(parents=True, exist_ok=True)
if not ASPAN_WEIGHTS_LOCAL.exists():
    shutil.copy(ASPAN_WEIGHTS_DRIVE, ASPAN_WEIGHTS_LOCAL)
    print(f'Copied ASpan weights -> {ASPAN_WEIGHTS_LOCAL}')
else:
    print(f'ASpan weights already staged at {ASPAN_WEIGHTS_LOCAL}')

if str(LOCAL_ROOT) not in sys.path:
    sys.path.insert(0, str(LOCAL_ROOT))
print('sys.path ready.')

In [ ]:
# -- Build a small retrieval manifest containing exactly the reviewed pairs --------
import csv, json

def load_reviewed_pairs(csv_path):
    rows = []
    with open(csv_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            source_id = row['source_folder']
            target_id = Path(row['target_image']).stem  # CSV has an extension, manifest doesn't
            rows.append({
                'source_id': source_id,
                'target_id': target_id,
                'classification': row['classification'],
            })
    return rows

reviewed_s1 = load_reviewed_pairs(SHARD1_CSV)
reviewed_s2 = load_reviewed_pairs(SHARD2_CSV)
reviewed = reviewed_s1 + reviewed_s2
print(f'Reviewed pairs from CSVs: {len(reviewed)} ({len(reviewed_s1)} shard1 + {len(reviewed_s2)} shard2)')

print('Indexing full retrieval manifest ...')
retrieval_index = {}
with open(RETRIEVAL_MANIFEST, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        retrieval_index[(row['source_id'], row['target_id'])] = row
print(f'Indexed {len(retrieval_index)} candidate rows.')

matched_rows = []
classification_lookup = {}
unmatched = []
for pair in reviewed:
    key = (pair['source_id'], pair['target_id'])
    src_row = retrieval_index.get(key)
    if src_row is None:
        unmatched.append(pair)
        continue
    row = dict(src_row)
    row['source_path'] = str(LOCAL_SOURCE_DIR / f"{row['source_id']}.jpg")
    row['target_path'] = str(LOCAL_TARGET_DIR / f"{row['target_id']}.jpg")
    matched_rows.append(row)
    classification_lookup[row['candidate_id']] = pair['classification']

print(f'Matched to retrieval manifest: {len(matched_rows)} / {len(reviewed)}')
if unmatched:
    print(f'WARNING: {len(unmatched)} reviewed pairs had NO matching retrieval-manifest row -- not silently dropped, listing:')
    for pair in unmatched:
        print(f"   {pair['source_id']} -> {pair['target_id']}  (classification={pair['classification']})")
else:
    print('All reviewed pairs matched -- no gaps.')

REVIEWED_RETRIEVAL_MANIFEST = LOCAL_ROOT / 'reviewed_pairs_retrieval.jsonl'
with open(REVIEWED_RETRIEVAL_MANIFEST, 'w', encoding='utf-8') as f:
    for row in matched_rows:
        f.write(json.dumps(row) + '\n')
print(f'Wrote {len(matched_rows)} rows to {REVIEWED_RETRIEVAL_MANIFEST}')

CLASSIFICATION_LOOKUP_PATH = LOCAL_ROOT / 'candidate_classification.json'
with open(CLASSIFICATION_LOOKUP_PATH, 'w', encoding='utf-8') as f:
    json.dump(classification_lookup, f, indent=2)
print(f'Wrote classification side-lookup ({len(classification_lookup)} entries) to {CLASSIFICATION_LOOKUP_PATH}')

In [ ]:
# -- Selective image extraction -- only the images the reviewed pairs need, not the full ~45k corpus
import zipfile

def extract_specific_images(zip_path, wanted_names, dest_dir):
    dest_dir.mkdir(parents=True, exist_ok=True)
    already = {p.name for p in dest_dir.glob('*.jpg')}
    missing = wanted_names - already
    if not missing:
        print(f'{dest_dir.name}: all {len(wanted_names)} needed image(s) already present')
        return
    with zipfile.ZipFile(zip_path) as zf:
        by_basename = {}
        for name in zf.namelist():
            if '__MACOSX' in name or name.endswith('/'):
                continue
            by_basename.setdefault(Path(name).name, name)
        found = 0
        for fname in missing:
            internal = by_basename.get(fname)
            if internal is None:
                print(f'  [warn] {fname} not found in {zip_path.name}')
                continue
            with zf.open(internal) as src, open(dest_dir / fname, 'wb') as dst:
                dst.write(src.read())
            found += 1
    print(f'{dest_dir.name}: extracted {found} new image(s) '
          f'(of {len(missing)} missing, {len(wanted_names)} needed total)')

needed_sources = {f"{row['source_id']}.jpg" for row in matched_rows}
needed_targets = {f"{row['target_id']}.jpg" for row in matched_rows}
print(f'Need {len(needed_sources)} source image(s), {len(needed_targets)} target image(s)')
extract_specific_images(SOURCE_ZIP, needed_sources, LOCAL_SOURCE_DIR)
extract_specific_images(TARGET_ZIP, needed_targets, LOCAL_TARGET_DIR)

In [ ]:
# -- Run ASpanFormer (unmodified geometry_filter.py) on the reviewed-pairs manifest --
import os, importlib

os.chdir(LOCAL_ROOT)
import geometry_filter as aspanfilter
importlib.reload(aspanfilter)

aspanfilter.main([
    '--input-manifest', str(REVIEWED_RETRIEVAL_MANIFEST),
    '--output-dir',     str(ASPAN_OUTPUT_DIR),
    '--aspanpath',      str(ASPAN_ROOT),
    '--weights_path',   str(ASPAN_WEIGHTS_LOCAL),
    '--config_path',    str(ASPAN_CONFIG),
    '--resume',
    *(['--max-pairs', str(MAX_PAIRS)] if MAX_PAIRS else []),
])

In [ ]:
# -- Verify: expected vs actual, report don't silently drop --
import json

def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open(encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())

all_manifest = ASPAN_OUTPUT_DIR / 'aspan_all_manifest.jsonl'
passed_manifest = ASPAN_OUTPUT_DIR / 'vggt_candidates_manifest.jsonl'

n_input = len(matched_rows)
n_checked = count_jsonl(all_manifest)
n_passed = count_jsonl(passed_manifest)
print(f'Input reviewed pairs:        {n_input}')
print(f'Checked by ASpanFormer:      {n_checked}')
print(f'Passed (sidecar produced):   {n_passed}')

if MAX_PAIRS is None:
    expected_ids = {row['candidate_id'] for row in matched_rows}
    passed_ids = set()
    with open(passed_manifest, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                passed_ids.add(json.loads(line)['candidate_id'])
    missing_ids = expected_ids - passed_ids
    if missing_ids:
        print(f'\nWARNING: {len(missing_ids)} reviewed pair(s) did NOT pass Stage 2 this time '
              f'(historically all were survivors -- see plan notes on determinism):')
        for cid in sorted(missing_ids):
            label = classification_lookup.get(cid, '?')
            print(f'   {cid}  (classification={label})')
    else:
        print('\nAll reviewed pairs reproduced as Stage-2 survivors -- no gaps.')
else:
    print(f'\nMAX_PAIRS={MAX_PAIRS} set -- this was a smoke test, gap-checking skipped. '
          f'Set MAX_PAIRS = None and re-run for the full pass.')

# Spot-check one sidecar
import numpy as np
with open(passed_manifest, encoding='utf-8') as f:
    first_line = f.readline()
if first_line.strip():
    first_row = json.loads(first_line)
    sidecar = np.load(first_row['sidecar_path'])
    print(f"\nSpot check {first_row['candidate_id']}:")
    print(f"  raw_mkpts0_resized shape:      {sidecar['raw_mkpts0_resized'].shape}")
    print(f"  filtered_mkpts0_resized shape: {sidecar['filtered_mkpts0_resized'].shape}")
else:
    print('\nNo passed candidates yet -- nothing to spot-check.')

---
## Part 2 -- MASt3R inference, threshold derivation, and scoring (B10)

Runs MASt3R on the reviewed pairs, using the exact homography-aligned pair VGGT would
see (per `ablation/MAST3R_ABLATION_METHODOLOGY.md`), derives a decision threshold on
Shard 1 only, then scores Shard 1 (dev) and Shard 2 (validation) and compares against
the existing VGGT (B4) baseline via McNemar (`ablation/significance.py`'s exact-binomial
implementation -- see `ablation/STATISTICS_METHODOLOGY.md`).

**Before running this section**, additionally have on Drive:
- `ablation/mast3r_signals.py` at `[DRIVE_ROOT]/scripts/mast3r_signals.py`
- `vggt_signals.py` at `[DRIVE_ROOT]/vggt_signals.py` (already there for the main pipeline)
- `ablation/significance.py` at `[DRIVE_ROOT]/scripts/significance.py`
- `pose_scoring.py` at `[DRIVE_ROOT]/pose_scoring.py`
- `aspan_all_manifest_shard1.jsonl`, `aspan_all_manifest_shard2.jsonl` at
  `[DRIVE_ROOT]/pipeline_output/` -- the canonical B4 baseline source this whole project
  uses (same file `ablation/statistics.py` and `ablation/eval_stage2.py` read; no longer
  the `Shard1/2 Judge Manifest.jsonl` files this notebook previously hardcoded a threshold
  rule against)
- **(Recommended) the MASt3R checkpoint, pre-staged by hand** at
  `[DRIVE_ROOT]/weights/mast3r_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth` --
  the live Hugging Face Hub download inside `from_pretrained()` has been unreliable
  (stalls/restarts from 0%). To pre-stage: grab the matching `.pth` link for the
  `MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric` variant from MASt3R's GitHub
  README checkpoints section, download it in your browser (resumable, bypasses
  Colab's network entirely), then upload that single file to the Drive path above.
  `cell-copy-mast3r-script` stages it to local disk automatically if present;
  `cell-run-mast3r` prefers it over the live download and loudly reports which one
  it used. If it's not there, the notebook still works -- it just falls back to the
  live HF download.

No composite MASt3R pose-score formula is hardcoded anywhere in this notebook or in
`mast3r_signals.py` -- the threshold-derivation cell below empirically compares candidate
formulas on Shard 1 only and freezes whichever performs best there, the same way the
existing VGGT pose-component weights were themselves derived rather than assumed.

**Output of this section, once run:** `[DRIVE_ROOT]/ablation_outputs/b10_mast3r/mast3r_stage3_results.json`
-- raw (uncorrected) McNemar tests for B10, ready for `ablation/aggregate_significance.py`
to fold into the stage3 family once downloaded locally to `D:/DINO OUTPUTS/`.

In [ ]:
# -- Install MASt3R -- no CUDA extension compilation needed for basic inference (RoPE
#    kernel compilation is optional/perf-only; confirmed from the real README).
import subprocess, sys
from pathlib import Path

def _run(*args):
    subprocess.run(list(args), check=True)

MAST3R_DIR = LOCAL_ROOT / 'mast3r'
if not MAST3R_DIR.exists():
    _run('git', 'clone', '--recursive', 'https://github.com/naver/mast3r', str(MAST3R_DIR))
else:
    print(f'{MAST3R_DIR} already present, skipping clone.')

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

_pip('-r', str(MAST3R_DIR / 'requirements.txt'))
_pip('-r', str(MAST3R_DIR / 'dust3r' / 'requirements.txt'))

if str(MAST3R_DIR) not in sys.path:
    sys.path.insert(0, str(MAST3R_DIR))
print('MASt3R install done.')

In [ ]:
import shutil, sys

MAST3R_SIGNALS_SCRIPT_DRIVE = DRIVE_ROOT / 'scripts' / 'mast3r_signals.py'
VGGT_SIGNALS_SCRIPT_DRIVE = DRIVE_ROOT / 'vggt_signals.py'

# mast3r_signals.py imports vggt_signals.py as a sibling module (repo layout:
# vggt_signals.py at repo root, ablation/mast3r_signals.py one level down) -- mirror that
# layout locally so the same sys.path.insert(0, parent.parent) trick inside
# mast3r_signals.py resolves correctly.
MAST3R_WORK_ROOT = LOCAL_ROOT / 'repo'
ABLATION_DIR = MAST3R_WORK_ROOT / 'ablation'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(VGGT_SIGNALS_SCRIPT_DRIVE, MAST3R_WORK_ROOT / 'vggt_signals.py')
shutil.copy(MAST3R_SIGNALS_SCRIPT_DRIVE, ABLATION_DIR / 'mast3r_signals.py')
print(f'Staged vggt_signals.py -> {MAST3R_WORK_ROOT}')
print(f'Staged mast3r_signals.py -> {ABLATION_DIR}')

if str(ABLATION_DIR) not in sys.path:
    sys.path.insert(0, str(ABLATION_DIR))
print('sys.path ready.')

# -- Stage the pre-downloaded MASt3R checkpoint locally, if present on Drive --------
# (see cell-mast3r-header for how to get the .pth there by hand). Copying to local
# Colab disk first, same pattern as ASPAN_WEIGHTS_LOCAL above, avoids repeatedly
# reading a ~2.75GB file over the Drive FUSE mount.
MAST3R_CKPT_LOCAL.parent.mkdir(parents=True, exist_ok=True)
if MAST3R_CKPT_LOCAL.exists():
    print(f'MASt3R checkpoint already staged at {MAST3R_CKPT_LOCAL}')
elif MAST3R_CKPT_DRIVE.exists():
    shutil.copy(MAST3R_CKPT_DRIVE, MAST3R_CKPT_LOCAL)
    print(f'Copied MASt3R checkpoint -> {MAST3R_CKPT_LOCAL}')
else:
    print(f'No pre-staged MASt3R checkpoint found at {MAST3R_CKPT_DRIVE} -- '
          f'cell-run-mast3r will fall back to a live Hugging Face Hub download instead.')

In [ ]:
# -- Run MASt3R on the reviewed pairs (uses the regenerated ASpanFormer sidecars) --
import os, importlib

os.chdir(LOCAL_ROOT)
import mast3r_signals
importlib.reload(mast3r_signals)

MAST3R_OUTPUT_DIR = LOCAL_ROOT / 'mast3r_output'

# Prefer the pre-staged local .pth (see cell-copy-mast3r-script) over a live Hugging
# Face Hub download -- loudly reported either way, never a silent behavior switch.
if MAST3R_CKPT_LOCAL.exists():
    mast3r_checkpoint_arg = str(MAST3R_CKPT_LOCAL)
    print(f'Using pre-staged local checkpoint: {mast3r_checkpoint_arg}')
else:
    mast3r_checkpoint_arg = 'naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric'
    print(f'No local checkpoint staged at {MAST3R_CKPT_LOCAL} -- falling back to a live '
          f'Hugging Face Hub download of {mast3r_checkpoint_arg} (see cell-mast3r-header '
          f'for how to pre-stage a .pth on Drive instead).')

mast3r_signals.main([
    '--input-manifest', str(ASPAN_OUTPUT_DIR / 'vggt_candidates_manifest.jsonl'),
    '--output-dir',     str(MAST3R_OUTPUT_DIR),
    '--checkpoint',     mast3r_checkpoint_arg,
    '--resume',
    *(['--max-pairs', str(MAX_PAIRS)] if MAX_PAIRS else []),
])

In [ ]:
# -- Derive a decision threshold for MASt3R's pose signal on Shard 1 ONLY --
# Mirrors how the existing VGGT pose-component weights (Table B) were themselves
# derived: an empirical sweep on the dev shard, frozen before ever touching Shard 2/3.
import csv, json

def shard_candidate_ids(csv_path, index):
    ids = set()
    with open(csv_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            source_id = row['source_folder']
            target_id = Path(row['target_image']).stem
            src_row = index.get((source_id, target_id))
            if src_row is not None:
                ids.add(src_row['candidate_id'])
    return ids

shard1_ids = shard_candidate_ids(SHARD1_CSV, retrieval_index)
shard2_ids = shard_candidate_ids(SHARD2_CSV, retrieval_index)
print(f'Shard 1: {len(shard1_ids)} candidate_ids; Shard 2: {len(shard2_ids)} candidate_ids')

FILTER2_THRESHOLD = 0.65  # unchanged from B4 -- not being ablated

def load_jsonl_dict(path, key='candidate_id'):
    out = {}
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                row = json.loads(line)
                out[row[key]] = row
    return out

MAST3R_JUDGED_PATH = MAST3R_OUTPUT_DIR / 'mast3r_judged_manifest.jsonl'
mast3r_judged = load_jsonl_dict(MAST3R_JUDGED_PATH)
print(f'MASt3R judged rows: {len(mast3r_judged)}')

def true_label(classification):
    if classification == 'Positive':
        return True
    if classification == 'Negative':
        return False
    return None  # Unsure / Unknown -- excluded from threshold fitting, never dropped from the manifest

shard1_dev = []
for cid in shard1_ids:
    label = true_label(classification_lookup.get(cid))
    if label is None:
        continue
    row = mast3r_judged.get(cid)
    if row is None or row.get('error'):
        continue
    inlier_ratio = row.get('aspan_2d_inlier_ratio')
    if inlier_ratio is None or inlier_ratio < FILTER2_THRESHOLD:
        continue  # Filter 2 already rejects this candidate regardless of the pose signal
    shard1_dev.append((cid, label, row))

print(f'Shard 1 dev set for threshold fitting (Positive/Negative, Filter-2-passing, MASt3R-scored): {len(shard1_dev)}')

def prf1_at_threshold(rows, score_fn, threshold):
    tp = fp = fn = 0
    for _cid, label, row in rows:
        score = score_fn(row)
        if score is None:
            continue
        predicted = score <= threshold
        if predicted and label:
            tp += 1
        elif predicted and not label:
            fp += 1
        elif not predicted and label:
            fn += 1
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1

def score_rotation_only(row):
    return row.get('mast3r_rotation_angle_deg')

def score_translation_only(row):
    return row.get('mast3r_pnp_translation_scaled')

def score_rotation_plus_translation(row, weight):
    rotation = row.get('mast3r_rotation_angle_deg')
    translation = row.get('mast3r_pnp_translation_scaled')
    if rotation is None or translation is None:
        return None
    return rotation + weight * translation

# translation_only added 2026-07-14: the essential-matrix route (rotation signal) was
# found to degenerate to exactly 180 deg for ~5% of candidates -- disproportionately
# (35x) on true-copy pairs, a known near-zero-baseline singularity in cv2.recoverPose's
# R/t decomposition. Rather than hand-picking translation-only as "the fix", it's added
# here as one more candidate in the same disciplined empirical sweep, so it's only
# selected if it actually wins on Shard 1 dev -- same process, wider search space.
candidate_formulas = {'rotation_only': score_rotation_only, 'translation_only': score_translation_only}
for weight in [0.01, 0.05, 0.1, 0.5, 1.0]:
    candidate_formulas[f'rotation_plus_{weight}_translation'] = (
        lambda row, weight=weight: score_rotation_plus_translation(row, weight)
    )

best = None
for formula_name, score_fn in candidate_formulas.items():
    candidate_scores = sorted({s for _, _, row in shard1_dev if (s := score_fn(row)) is not None})
    for i in range(len(candidate_scores) - 1):
        threshold = (candidate_scores[i] + candidate_scores[i + 1]) / 2
        precision, recall, f1 = prf1_at_threshold(shard1_dev, score_fn, threshold)
        if best is None or f1 > best['f1']:
            best = {
                'formula': formula_name, 'threshold': threshold,
                'precision': precision, 'recall': recall, 'f1': f1,
            }

print()
print('=' * 70)
print('FROZEN on Shard 1 dev ONLY -- never re-tuned on Shard 2/3:')
print(f"  formula   = {best['formula']}")
print(f"  threshold = {best['threshold']:.4f}")
print(f"  Shard 1 dev P/R/F1 = {best['precision']:.3f} / {best['recall']:.3f} / {best['f1']:.3f}")
print('=' * 70)

FROZEN_FORMULA_NAME = best['formula']
FROZEN_SCORE_FN = candidate_formulas[FROZEN_FORMULA_NAME]
FROZEN_THRESHOLD = best['threshold']

In [ ]:
# -- Apply the frozen formula+threshold to Shard 1 (dev) and Shard 2 (validation) --
# Filter 2 (aspan_2d_inlier_ratio >= 0.65) stays exactly as in B4; only Filter 3 (pose)
# is replaced by the MASt3R-derived score+threshold frozen in the previous cell.
import random

def score_shard(shard_ids):
    rows = []
    for cid in shard_ids:
        label = true_label(classification_lookup.get(cid))
        if label is None:
            continue
        row = mast3r_judged.get(cid)
        if row is None or row.get('error'):
            continue
        inlier_ratio = row.get('aspan_2d_inlier_ratio')
        if inlier_ratio is None:
            continue
        filter2_pass = inlier_ratio >= FILTER2_THRESHOLD
        score = FROZEN_SCORE_FN(row)
        filter3_pass = (score is not None) and (score <= FROZEN_THRESHOLD)
        predicted = filter2_pass and filter3_pass
        rows.append({'candidate_id': cid, 'label': label, 'predicted': predicted})
    return rows

def prf1(rows):
    tp = sum(1 for r in rows if r['predicted'] and r['label'])
    fp = sum(1 for r in rows if r['predicted'] and not r['label'])
    fn = sum(1 for r in rows if not r['predicted'] and r['label'])
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1

def bootstrap_ci(rows, n_resamples=10000, seed=0):
    rng = random.Random(seed)
    n = len(rows)
    if n == 0:
        return 0.0, 0.0
    f1_samples = []
    for _ in range(n_resamples):
        sample = [rows[rng.randrange(n)] for _ in range(n)]
        _, _, f1 = prf1(sample)
        f1_samples.append(f1)
    f1_samples.sort()
    lo = f1_samples[int(0.025 * n_resamples)]
    hi = f1_samples[int(0.975 * n_resamples)]
    return lo, hi

shard_results = {}
for shard_name, shard_ids in [('Shard 1 (dev)', shard1_ids), ('Shard 2 (validation)', shard2_ids)]:
    rows = score_shard(shard_ids)
    precision, recall, f1 = prf1(rows)
    lo, hi = bootstrap_ci(rows)
    shard_results[shard_name] = rows
    print(f'{shard_name}: n={len(rows)}  P={precision:.3f}  R={recall:.3f}  F1={f1:.3f}  95% CI=[{lo:.3f}, {hi:.3f}]')

print()
print('Shard 1 numbers above are the dev set the threshold was fit on -- report them as')
print('such, not as an independent result. Shard 2 is the real reported comparison.')

In [ ]:
# -- McNemar vs the canonical B4 baseline, on the same candidates --
# Uses ablation/significance.py (the same exact-binomial McNemar implementation
# ablation/statistics.py and ablation/eval_stage2.py import -- no reimplementation) and
# the canonical B4 source, aspan_all_manifest_shard{N}.jsonl via pose_scoring.score_row()
# with its default thresholds (not a hardcoded inlier/pose-score rule -- see
# ablation/STATISTICS_METHODOLOGY.md). Joins on (source_id, target_id), matching every
# other script -- not candidate_id, which is local to this notebook's own indexing.
import shutil, sys, json

SIGNIFICANCE_SCRIPT_DRIVE = DRIVE_ROOT / 'scripts' / 'significance.py'
POSE_SCORING_SCRIPT_DRIVE = DRIVE_ROOT / 'pose_scoring.py'

shutil.copy(SIGNIFICANCE_SCRIPT_DRIVE, ABLATION_DIR / 'significance.py')
shutil.copy(POSE_SCORING_SCRIPT_DRIVE, ABLATION_DIR / 'pose_scoring.py')
print(f'Staged significance.py -> {ABLATION_DIR}')
print(f'Staged pose_scoring.py -> {ABLATION_DIR}')

from significance import mcnemar_exact
from pose_scoring import score_row, INLIER_RATIO_THRESHOLD, POSE_COMPONENT_THRESHOLD

ASPAN_ALL_S1 = DRIVE_ROOT / 'pipeline_output' / 'aspan_all_manifest_shard1.jsonl'
ASPAN_ALL_S2 = DRIVE_ROOT / 'pipeline_output' / 'aspan_all_manifest_shard2.jsonl'

def load_aspan_all_by_key(path):
    out = {}
    if not path.exists():
        print(f'WARNING: {path} not found -- upload it to run the B4 comparison')
        return out
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                row = json.loads(line)
                out[(row['source_id'], row['target_id'])] = row
    return out

aspan_all_b4 = {}
aspan_all_b4.update(load_aspan_all_by_key(ASPAN_ALL_S1))
aspan_all_b4.update(load_aspan_all_by_key(ASPAN_ALL_S2))
print(f'Canonical B4 source (aspan_all_manifest_shard1+2.jsonl): {len(aspan_all_b4)} rows')

# Same defaults ablation/statistics.py's B4 uses -- imported constants, not re-typed
# numbers, so this can never silently drift from the canonical source.
B4_SCORE_DEFAULTS = dict(
    inlier_ratio_threshold=INLIER_RATIO_THRESHOLD,
    pose_component_threshold=POSE_COMPONENT_THRESHOLD,
    global_sim_threshold=None,
    pose_components='all',
    keypoint_floor=0,
)

def b4_predicted(key):
    row = aspan_all_b4.get(key)
    if row is None or 'aspan_2d_inlier_ratio' not in row:
        return None
    predicted, _reason = score_row(row, **B4_SCORE_DEFAULTS)
    return bool(predicted)

# candidate_id -> (source_id, target_id), built from matched_rows (cell-build-manifest)
# so this join never depends on candidate_id surviving unchanged through
# geometry_filter.py/mast3r_signals.py -- it's read straight from the original
# retrieval-manifest row every candidate_id was assigned from.
cid_to_key = {row['candidate_id']: (row['source_id'], row['target_id']) for row in matched_rows}

def mcnemar_test_for_shard(rows_b10, shard_label):
    b = c = 0  # b = B10 correct & B4 wrong; c = B4 correct & B10 wrong
    n_paired = 0
    for r in rows_b10:
        key = cid_to_key.get(r['candidate_id'])
        if key is None:
            continue
        b4_pred = b4_predicted(key)
        if b4_pred is None:
            continue
        n_paired += 1
        b10_correct = (r['predicted'] == r['label'])
        b4_correct = (b4_pred == r['label'])
        if b10_correct and not b4_correct:
            b += 1
        elif b4_correct and not b10_correct:
            c += 1
    test = mcnemar_exact(b, c)
    test['row'] = 'B10'
    test['shard'] = shard_label
    test['family'] = 'stage3'
    test['n_paired'] = n_paired
    return test

SHARD_LABELS = {'Shard 1 (dev)': 'Shard1', 'Shard 2 (validation)': 'Shard2'}

mast3r_tests = []
for shard_name, rows in shard_results.items():
    test = mcnemar_test_for_shard(rows, SHARD_LABELS[shard_name])
    mast3r_tests.append(test)
    print(f"{shard_name}: McNemar b={test['b']} c={test['c']} (n_paired={test['n_paired']})  "
          f"raw_p={test['p_value']:.4f}  method={test['method']}  direction={test['direction']}")

print()
print('NOTE: raw p-values only, NOT yet Holm-corrected -- correction happens once these')
print('rows join the rest of the stage3 family (B1/B2/B5/B6/B8/B11) in')
print('ablation/aggregate_significance.py, which this notebook has no visibility into on')
print('its own. mast3r_tests (below) is what cell-save-outputs syncs to Drive for that step.')

In [ ]:
# -- Save B10 outputs to Drive --
import json, shutil

B10_DRIVE_OUT = DRIVE_ROOT / 'ablation_outputs' / 'b10_mast3r'
B10_DRIVE_OUT.mkdir(parents=True, exist_ok=True)

src = MAST3R_OUTPUT_DIR / 'mast3r_judged_manifest.jsonl'
if src.exists():
    shutil.copy2(src, B10_DRIVE_OUT / src.name)
    print(f'Saved: {src.name}')
else:
    print(f'MISSING (not yet produced): {src.name}')

summary = {
    'frozen_formula': FROZEN_FORMULA_NAME,
    'frozen_threshold': FROZEN_THRESHOLD,
    'shard1_dev_n': len(shard1_dev),
    'shard_results_n': {name: len(rows) for name, rows in shard_results.items()},
}
summary_path = B10_DRIVE_OUT / 'b10_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print(f'Saved summary to {summary_path}')

# Raw (uncorrected) Stage 3 McNemar tests for B10, in the exact schema
# ablation/aggregate_significance.py's load_stage3_from_mast3r() expects: a JSON list
# of test dicts, each already carrying row="B10"/shard/family="stage3" (see cell-mcnemar).
# This is the file to download from Drive to D:/DINO OUTPUTS/mast3r_stage3_results.json
# so aggregate_significance.py can pick it up and Holm-correct the stage3 family with
# B10 included instead of reporting it PROVISIONAL.
mast3r_results_path = B10_DRIVE_OUT / 'mast3r_stage3_results.json'
with open(mast3r_results_path, 'w', encoding='utf-8') as f:
    json.dump(mast3r_tests, f, indent=2)
print(f'Saved raw Stage 3 McNemar tests to {mast3r_results_path}')
print('Download this file to D:/DINO OUTPUTS/mast3r_stage3_results.json, then re-run')
print('ablation/aggregate_significance.py.')